# Previsão do consumo horário de energia (PJM East) com LSTM — janela de 24h × 168h

**Disciplina:** Inteligência Computacional / Machine Learning — CC6NA (2026.2)<br>
**Professora:** Polyana Santos Fonseca Nascimento<br>
**Dupla:** Davi Corrêa e Gabriel Alencar

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaalbu/lstm-energia-pjme/blob/main/notebooks/LSTM_PJME_energia.ipynb)

**Objetivo:** prever o consumo de energia (MW) da **próxima hora** a partir do histórico recente da própria série, comparando
duas janelas deslizantes de entrada — **24 horas** (um dia) e **168 horas** (uma semana) — e verificando se o histórico mais
longo melhora a previsão.

| Seção | Conteúdo | Responsável |
|---|---|---|
| 0 | Declaração de uso de IA Generativa | Gabriel (rascunho) · Davi (revisão final) |
| 1–2 | Configuração do ambiente, dataset e gráfico inicial | Gabriel |
| 3 | Ordenação cronológica e tratamento de lacunas | Gabriel |
| 4 | Divisão treino/validação/teste e normalização | Gabriel |
| 5 | Construção das janelas deslizantes (24h e 168h) | Gabriel |
| 6 | Arquitetura e treino das duas LSTMs | Gabriel |
| 7 | MAE na validação e escolha da janela | Gabriel |
| 8 | Teste: LSTM escolhida × persistência × referência sazonal | Davi |
| 9 | Visualizações complementares do teste | Davi |
| 10 | Conclusão | Davi (com Gabriel) |

## 0. Declaração de uso de IA Generativa

> ✏️ **Revisar antes da entrega** — ajustar a extensão e as medidas de validação ao que a dupla de fato fez.

- **Houve uso de IA Generativa?** Sim.
- **Ferramenta utilizada:** Claude (Anthropic).
- **Finalidade do uso:** apoio na estruturação do notebook segundo a lauda, na escrita do código de pré-processamento,
  construção das janelas e treino das LSTMs, e na redação dos textos explicativos.
- **Extensão aproximada da contribuição:** _preencher (ex.: parte significativa do código das seções 1–7; seções 8–10 feitas pela dupla)_.
- **Medidas de validação:** execução completa do notebook do início ao fim; conferência manual das lacunas encontradas na
  série (horário de verão); verificação de que o normalizador foi ajustado só com o treino e de que as duas janelas usam
  exatamente as mesmas datas-alvo (asserts no código); conferência dos MAE na escala original.

## 1. Configuração do ambiente

O notebook usa apenas bibliotecas que já vêm instaladas no Google Colab (pandas, NumPy, Matplotlib, scikit-learn e
TensorFlow/Keras). Recomenda-se usar um ambiente com **GPU** (*Ambiente de execução → Alterar o tipo de ambiente de
execução → T4 GPU*) — em CPU o treino da janela de 168h leva alguns minutos a mais, mas também funciona.

As sementes aleatórias são fixadas para que os resultados sejam reprodutíveis (podem variar levemente entre CPU e GPU).

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras

SEED = 42
keras.utils.set_random_seed(SEED)

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "não")

## 2. Dataset

- **Nome:** *Hourly Energy Consumption* — arquivo `PJME_hourly.csv`
- **Fonte:** Kaggle — <https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption>. Os dados são publicados
  originalmente pela **PJM Interconnection**, operadora da rede elétrica de parte do leste dos EUA; `PJME` é a região
  *PJM East*.
- **Variável-alvo:** `PJME_MW` — consumo (carga) de energia elétrica em megawatts.
- **Frequência:** horária, de 01/01/2002 a 03/08/2018 (≈ 145 mil observações).
- **Problema:** prever o consumo da **próxima hora** (um passo à frente) usando apenas os valores anteriores da série.

**Por que esse dataset?** Previsão de carga de curto prazo é um problema real das operadoras (despacho de usinas,
compra de energia). A série é longa, vem de uma fonte confiável e tem ciclos fortes e bem conhecidos — diário (dia × noite),
semanal (dias úteis × fim de semana) e anual (verão e inverno) — o que a torna adequada para testar se uma janela de
uma semana captura algo que uma janela de um dia não captura.

Para que o notebook rode sem precisar de conta no Kaggle, uma cópia do CSV original está na pasta `data/` do repositório.

In [ ]:
CAMINHO_LOCAL = "../data/PJME_hourly.csv"
URL_REPO = "https://raw.githubusercontent.com/Gaalbu/lstm-energia-pjme/main/data/PJME_hourly.csv"

fonte = CAMINHO_LOCAL if os.path.exists(CAMINHO_LOCAL) else URL_REPO
bruto = pd.read_csv(fonte, parse_dates=["Datetime"])
print("Fonte:", fonte)
print(bruto.shape)
bruto.head()

In [ ]:
bruto.describe()

## 3. Ordem cronológica e tratamento de lacunas

O CSV **não vem em ordem cronológica** (começa em 31/12/2002), então o primeiro passo é ordenar pelas datas. Depois
verificamos se a série tem exatamente uma observação por hora.

In [ ]:
bruto = bruto.sort_values("Datetime").reset_index(drop=True)

duplicadas = bruto[bruto["Datetime"].duplicated(keep=False)]
grade_completa = pd.date_range(bruto["Datetime"].min(), bruto["Datetime"].max(), freq="h")
faltantes = grade_completa.difference(bruto["Datetime"])

print(f"Valores nulos em PJME_MW: {bruto['PJME_MW'].isna().sum()}")
print(f"Horários repetidos: {duplicadas['Datetime'].nunique()}")
print(f"Horas ausentes: {len(faltantes)}")
print("\nRepetidos:")
print(duplicadas.to_string(index=False))
print("\nAusentes (amostra):", [str(t) for t in faltantes[:6]], "...")
print("Meses das horas ausentes:", sorted(set(faltantes.month)))

As inconsistências vêm, em sua maioria, do **horário de verão** dos EUA:

- **Horas repetidas** (4 casos, em novembro): quando o relógio volta uma hora, o horário das 02:00 aparece duas vezes.
  Tratamento: substituímos as duas leituras pela **média** delas — fica uma observação por hora.
- **Horas ausentes** (30 casos): a maioria cai nos dias de troca do horário de verão (março/abril e outubro/novembro) e há
  poucas lacunas isoladas em outras datas. Tratamento: **interpolação linear no tempo** entre a hora anterior e a seguinte.
  Como todas as lacunas são de 1 hora e o consumo varia de forma suave de uma hora para a outra, a interpolação é uma estimativa razoável
  e afeta apenas ~0,02% da série.

Depois disso a série fica regular (frequência horária, sem buracos), requisito para que cada janela deslizante de *k*
observações corresponda exatamente a *k* horas.

In [ ]:
serie = (bruto.groupby("Datetime")["PJME_MW"].mean()   # junta horários repetidos pela média
              .reindex(grade_completa)                   # cria as horas ausentes como NaN
              .interpolate(method="time"))               # preenche por interpolação linear no tempo
serie.index.name = "Datetime"
serie.name = "PJME_MW"

maior_lacuna = (pd.Series(faltantes).diff() != pd.Timedelta("1h")).cumsum().value_counts().max()
assert serie.isna().sum() == 0 and serie.index.is_monotonic_increasing and serie.index.freq == "h"
print(f"Série final: {len(serie)} horas | maior lacuna consecutiva preenchida: {maior_lacuna} h")

### Gráfico inicial da série

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(13, 10))

serie.plot(ax=ax[0], lw=0.3, color="tab:blue")
serie.resample("W").mean().plot(ax=ax[0], color="black", lw=1.2, label="média semanal")
ax[0].set_title("PJME — consumo horário completo (2002–2018)")
ax[0].set_ylabel("MW"); ax[0].legend()

semanas = serie["2017-07-03":"2017-07-16"]
semanas.plot(ax=ax[1], color="tab:orange")
ax[1].set_title("Zoom: duas semanas de julho/2017 — ciclo diário e diferença dias úteis × fim de semana")
ax[1].set_ylabel("MW")

perfil = serie.groupby([serie.index.dayofweek, serie.index.hour]).mean().unstack(0)
perfil.columns = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
perfil.plot(ax=ax[2])
ax[2].set_title("Perfil médio por hora do dia, para cada dia da semana")
ax[2].set_xlabel("hora do dia"); ax[2].set_ylabel("MW médio")

plt.tight_layout(); plt.show()

O gráfico mostra (1) uma **sazonalidade anual** com picos no verão (ar-condicionado) e um pico menor no inverno;
(2) um **ciclo diário** muito marcado — mínimo de madrugada e máximo à tarde/início da noite; e (3) um **padrão semanal**:
sábado e domingo têm consumo visivelmente menor que os dias úteis. Esses ciclos orientam a escolha das janelas na seção 5.

## 4. Divisão cronológica e normalização

A divisão é feita **pelas datas-alvo** (o instante que se quer prever), em ordem cronológica e sem embaralhar:

| Partição | Proporção da série | Uso |
|---|---|---|
| Treino | primeiros ~65% | ajustar os pesos da LSTM e o normalizador |
| Validação | ~15% seguintes | *early stopping* e **escolha da janela** (24h × 168h) |
| Teste | **últimos 20%** | avaliação final contra as referências |

Para que as duas configurações sejam avaliadas **nas mesmas datas-alvo**, o primeiro alvo possível é a hora 168 da série
(a janela maior precisa de 168 horas anteriores). Assim, a janela de 24h não ganha alvos "extras" no começo do treino.
A janela de entrada de um alvo pode conter horas da partição anterior (por exemplo, o primeiro alvo da validação usa as
últimas horas do treino como entrada) — isso não é vazamento, porque são valores passados, conhecidos no momento da
previsão.

A normalização Min-Max (escala 0–1) é **ajustada apenas com os valores do período de treino** e depois aplicada à série
inteira.

In [ ]:
JANELAS = [24, 168]
JANELA_MAX = max(JANELAS)

n = len(serie)
i_val = int(n * 0.65)    # primeiro alvo da validação
i_teste = int(n * 0.80)  # primeiro alvo do teste  -> últimos 20% da série

idx_alvos = {
    "treino": np.arange(JANELA_MAX, i_val),
    "validacao": np.arange(i_val, i_teste),
    "teste": np.arange(i_teste, n),
}
datas = {p: serie.index[i] for p, i in idx_alvos.items()}

for p, d in datas.items():
    print(f"{p:<10} {d[0]}  ->  {d[-1]}   ({len(d):>6} alvos, {len(d)/n:.1%} da série)")

In [ ]:
scaler = MinMaxScaler()
scaler.fit(serie.iloc[:i_val].to_frame())          # somente o período de treino
serie_norm = scaler.transform(serie.to_frame()).ravel().astype("float32")

print(f"Min/max usados pelo scaler (treino): {scaler.data_min_[0]:.0f} / {scaler.data_max_[0]:.0f} MW")
print(f"Faixa normalizada em validação+teste: {serie_norm[i_val:].min():.3f} a {serie_norm[i_val:].max():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
cores = {"treino": "tab:blue", "validacao": "tab:orange", "teste": "tab:green"}
for p, idx in idx_alvos.items():
    ax.plot(serie.index[idx], serie.values[idx], lw=0.3, color=cores[p], label=p)
ax.set_title("Divisão cronológica por data-alvo"); ax.set_ylabel("MW")
ax.legend(loc="upper left", markerscale=10); plt.show()

## 5. Janelas deslizantes de entrada

Usamos a **janela deslizante** de tamanho fixo: para prever o valor da hora *t*, a entrada são as *k* horas imediatamente
anteriores, $x_{t-k}, \dots, x_{t-1}$. A cada novo exemplo a janela avança **uma observação** (o início e o fim andam juntos).

**Tamanhos escolhidos (dados horários):**

- **k = 24 horas (um dia):** cobre um ciclo diário completo — a rede vê o mesmo horário do dia anterior e todo o formato
  da curva de consumo do último dia.
- **k = 168 horas (uma semana, 7 × 24):** cobre um ciclo semanal completo — além do dia anterior, a rede vê o mesmo
  horário e o mesmo dia da semana anterior, o que pode ajudar nas transições entre dias úteis e fim de semana
  (ex.: prever uma segunda-feira olhando só para um domingo tende a subestimar o consumo).

A pergunta do trabalho é se esse histórico extra (7× maior) compensa o custo de uma sequência mais longa.

In [ ]:
def criar_janelas(serie_norm, idx_alvos, k):
    # X[i] = os k valores anteriores ao alvo idx_alvos[i];  y[i] = valor no alvo
    offsets = np.arange(-k, 0)
    X = serie_norm[idx_alvos[:, None] + offsets[None, :]]
    y = serie_norm[idx_alvos]
    return X[..., None], y          # (amostras, k, 1 variável), (amostras,)

dados = {k: {p: criar_janelas(serie_norm, idx, k) for p, idx in idx_alvos.items()} for k in JANELAS}

for k in JANELAS:
    print(f"janela {k:>3}h:", {p: X.shape for p, (X, _) in dados[k].items()})

# conferências: mesmos alvos nas duas janelas e janela termina na hora anterior ao alvo
for p in idx_alvos:
    assert np.array_equal(dados[24][p][1], dados[168][p][1])
    assert np.array_equal(dados[24][p][0][:, :, 0], dados[168][p][0][:, -24:, 0])
i0 = idx_alvos["validacao"][0]
assert dados[24]["validacao"][0][0, -1, 0] == serie_norm[i0 - 1]
print("OK: as duas configurações têm exatamente as mesmas datas-alvo em todas as partições.")

## 6. Arquitetura e treino das LSTMs

As duas configurações usam **a mesma arquitetura e os mesmos hiperparâmetros** — a única diferença é o comprimento da
sequência de entrada:

- Entrada: sequência de *k* passos com 1 variável (consumo normalizado).
- Camada **LSTM com 64 unidades** (retorna só o último estado oculto).
- Camada **Dense(1)** linear → consumo previsto para a próxima hora (normalizado).
- Perda MSE, otimizador Adam (lr = 0,001), lotes de 256 exemplos, até 60 épocas.
- **Early stopping** monitorando a perda na validação (paciência 5), restaurando os melhores pesos.

O MSE é usado no treino por penalizar mais os erros grandes (picos de carga); a comparação entre janelas é feita com o
**MAE na escala original (MW)**, como pede a lauda.

In [ ]:
UNIDADES = 64
EPOCAS = 60
LOTE = 256
TAXA_APRENDIZADO = 1e-3
PACIENCIA = 5

def criar_lstm(k):
    modelo = keras.Sequential([
        keras.Input(shape=(k, 1)),
        keras.layers.LSTM(UNIDADES),
        keras.layers.Dense(1),
    ], name=f"lstm_janela_{k}h")
    modelo.compile(optimizer=keras.optimizers.Adam(TAXA_APRENDIZADO), loss="mse", metrics=["mae"])
    return modelo

criar_lstm(24).summary()

In [ ]:
modelos, historicos = {}, {}

for k in JANELAS:
    keras.utils.set_random_seed(SEED)       # mesma inicialização para as duas configurações
    X_tr, y_tr = dados[k]["treino"]
    X_va, y_va = dados[k]["validacao"]
    modelo = criar_lstm(k)
    parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True)
    print(f"\n=== Treinando janela de {k}h ===")
    hist = modelo.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=EPOCAS, batch_size=LOTE,
                      callbacks=[parada], verbose=2)
    modelos[k], historicos[k] = modelo, hist.history
    print(f"melhor época: {int(np.argmin(hist.history['val_loss'])) + 1} de {len(hist.history['val_loss'])}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for a, k in zip(ax, JANELAS):
    a.plot(historicos[k]["loss"], label="treino")
    a.plot(historicos[k]["val_loss"], label="validação")
    a.set_yscale("log"); a.set_title(f"Curva de perda (MSE normalizado) — janela {k}h")
    a.set_xlabel("época"); a.legend()
plt.tight_layout(); plt.show()

## 7. Comparação das janelas na validação (MAE em MW)

As previsões são convertidas de volta para MW com o mesmo `scaler` (transformação inversa) e o MAE é calculado na
**escala original**.

In [ ]:
def para_mw(valores_norm):
    return scaler.inverse_transform(np.asarray(valores_norm).reshape(-1, 1)).ravel()

def prever_mw(modelo, X):
    return para_mw(modelo.predict(X, batch_size=2048, verbose=0))

def mae(real, previsto):
    return float(np.mean(np.abs(np.asarray(real) - np.asarray(previsto))))

y_val_mw = serie.values[idx_alvos["validacao"]]
prev_val = {k: prever_mw(modelos[k], dados[k]["validacao"][0]) for k in JANELAS}
mae_val = {k: mae(y_val_mw, prev_val[k]) for k in JANELAS}

janela_escolhida = min(mae_val, key=mae_val.get)

tabela_val = pd.DataFrame({
    "Janela": [f"{k} h" for k in JANELAS],
    "MAE validação (MW)": [round(mae_val[k], 1) for k in JANELAS],
    "MAE / consumo médio": [f"{mae_val[k] / y_val_mw.mean():.2%}" for k in JANELAS],
    "Épocas treinadas": [len(historicos[k]["loss"]) for k in JANELAS],
    "Escolhida": ["✔" if k == janela_escolhida else "" for k in JANELAS],
})
display(tabela_val) if "display" in dir() else print(tabela_val)
print(f"\nJanela escolhida pela validação: {janela_escolhida} h")

In [ ]:
inicio, fim = "2014-07-07", "2014-07-20"     # duas semanas de verão (período de validação)
mask = (datas["validacao"] >= inicio) & (datas["validacao"] < fim)

fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax[0].plot(datas["validacao"][mask], y_val_mw[mask], color="black", lw=1.4, label="real")
for k, cor in zip(JANELAS, ["tab:blue", "tab:red"]):
    ax[0].plot(datas["validacao"][mask], prev_val[k][mask], color=cor, lw=1, alpha=0.85,
               label=f"LSTM {k}h (MAE período = {mae(y_val_mw[mask], prev_val[k][mask]):.0f} MW)")
    ax[1].plot(datas["validacao"][mask], prev_val[k][mask] - y_val_mw[mask], color=cor, lw=0.9, label=f"erro {k}h")
ax[0].set_title("Validação — real × previsões das duas janelas (zoom em duas semanas)"); ax[0].set_ylabel("MW"); ax[0].legend()
ax[1].axhline(0, color="black", lw=0.8); ax[1].set_ylabel("previsto − real (MW)"); ax[1].legend()
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
plt.tight_layout(); plt.show()

In [ ]:
erro_abs = pd.DataFrame({f"{k}h": np.abs(prev_val[k] - y_val_mw) for k in JANELAS}, index=datas["validacao"])
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
erro_abs.groupby(erro_abs.index.hour).mean().plot(ax=ax[0], marker="o", ms=3)
ax[0].set_title("MAE na validação por hora do dia"); ax[0].set_xlabel("hora"); ax[0].set_ylabel("MW")
por_dia = erro_abs.groupby(erro_abs.index.dayofweek).mean()
por_dia.index = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
por_dia.plot.bar(ax=ax[1], rot=0)
ax[1].set_title("MAE na validação por dia da semana"); ax[1].set_ylabel("MW")
plt.tight_layout(); plt.show()

### Leitura dos resultados da validação

> ✏️ **Escrever depois de rodar no Colab:** usar a tabela acima e a frase impressa na célula abaixo para comentar qual
> janela teve menor MAE, a diferença percentual, as curvas de perda (se houve sinal de *overfitting* ou se a rede ainda
> estava melhorando) e em quais horas/dias o erro foi maior. Numa execução preliminar (30 épocas, CPU), a janela de **168h**
> teve MAE ≈ 275 MW contra ≈ 308 MW da de 24h na validação.

In [ ]:
melhor, pior = janela_escolhida, [k for k in JANELAS if k != janela_escolhida][0]
ganho = (mae_val[pior] - mae_val[melhor]) / mae_val[pior]
print(f"A janela de {melhor}h teve MAE de {mae_val[melhor]:.1f} MW contra {mae_val[pior]:.1f} MW da janela de {pior}h "
      f"({ganho:.1%} menor). Ela é a configuração levada para o teste.")

---
### 🔁 Passagem para a parte do Davi

Tudo o que as seções 8–10 precisam já está em memória:

| Variável | O que é |
|---|---|
| `janela_escolhida` | 24 ou 168, escolhida pelo MAE da validação |
| `modelos[janela_escolhida]` | LSTM treinada da janela escolhida |
| `dados[janela_escolhida]["teste"]` | `(X_teste, y_teste_normalizado)` |
| `idx_alvos["teste"]`, `datas["teste"]` | posições e datas-alvo do teste |
| `serie` | série completa em MW (já limpa) — serve para as referências |
| `prever_mw(modelo, X)`, `mae(real, previsto)`, `para_mw(...)` | funções auxiliares |

Referências no teste (para o alvo na posição `t` de `serie`):
- **Persistência:** `serie.values[t - 1]` (último valor observado).
- **Sazonal diária:** `serie.values[t - 24]` (mesma hora do dia anterior) — ciclo diário justificado pelo gráfico da seção 2.
- (opcional) **Sazonal semanal:** `serie.values[t - 168]` (mesma hora da semana anterior).
---

## 8. Teste: LSTM escolhida × referências  — *parte do Davi*

> **TODO (Davi):**
> 1. Gerar as previsões da LSTM escolhida no teste com `prever_mw(modelos[janela_escolhida], dados[janela_escolhida]["teste"][0])`.
> 2. Calcular a persistência e a referência sazonal (24h) nas **mesmas datas-alvo** (`idx_alvos["teste"]`).
> 3. Montar a tabela com o MAE (MW) de cada método no teste.
> 4. Gráfico com valores reais + previsões da LSTM + cada referência (zoom de 1–2 semanas para ficar legível), com o MAE
>    de cada método na legenda.

In [ ]:
# TODO (Davi): seção 8 — avaliação no teste

## 9. Visualizações complementares  — *parte do Davi*

> **TODO (Davi):** sugestões — dispersão real × previsto da LSTM no teste; MAE por hora do dia / dia da semana comparando
> LSTM e referências; erro em feriados ou nos picos de verão.

In [ ]:
# TODO (Davi): seção 9 — visualizações extras

## 10. Conclusão  — *parte do Davi (com Gabriel)*

> **TODO:** responder (1) se a janela mais longa (168h) ajudou na validação, usando a tabela da seção 7; e (2) como a LSTM
> escolhida se saiu no teste em relação à persistência e à referência sazonal. Comentar limitações (apenas a própria série
> como entrada, sem temperatura/feriados; previsão de um único passo) e possíveis melhorias.